In [33]:
# Use only 1 GPU if available
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # Disable progress bars

from chronos import BaseChronosPipeline, Chronos2Pipeline
from datetime import datetime
import torch
import os
from datetime import datetime
from ts_toolkit.utils.chronos_tune_input_generator import chronos_input_generator


torch.cuda.empty_cache()


# Can we improve the bias?
Experiment: Finetune Chronos on Deterministic Trends datasets (constant, linear, exponential, staircase)
Measurement: Measure mean bias and R2 on test set of same datasets, and compare to base Chronos2. Does it get better?

### Data Generation ###

In [34]:
DISTRIBUTIONS = {
    "Deterministic Trends": ['constant', 'linear', 'exponential', 'staircase'],
    "Harmonic & Periodic": ['sine', 'sawtooth', 'square', 'triangle', 'noise_patch', 'fourier'],
    "Transient Dynamics": ['damped_sine', 'growing_sine', 'chirp'],
    "Autocorrelated Processes": ['ar', 'ma', 'arma', 'sarima', 'garch', 'pink_noise', 'fractional_brownian', 'markov_chain'], 
    "I.I.D. White Noise": ['normal', 'uniform', 'poisson', 'binary', 'student_t', 'lognormal', 'laplace', 'cauchy', 'skew_normal'],
    "Stochastic Drift/Shocks": ['random_walk', 'piecewise_constant', 'gbm', 'intermittent', 'impulse', 'logistic'], 
}

In [35]:
# List of level 1 dataset types, e.g., ["univariate_lagged", "multivariate_lagged"]. If None, all types are used.
dataset_level1 =None
# List of level 2 dataset types, e.g., ["exponential", "noise_patch", "normal", "poisson", "random_walk"]. If None, all types are used.
# all but Autocorrelated Processes
dataset_level2 = DISTRIBUTIONS["Deterministic Trends"]

data_path = "/home/nafiseh/GitHub/timeseries-research/data/simpletime/datasets_100k"
train_inputs, validation_inputs = chronos_input_generator(data_path, dataset_level1, dataset_level2, val_size=0.1)


multivariate_lagged/constant: 100000it [00:16, 6160.44it/s]
multivariate_lagged/linear: 100000it [00:17, 5842.51it/s]
multivariate_lagged/exponential: 100000it [00:17, 5757.92it/s]
multivariate_lagged/staircase: 100000it [00:17, 5729.48it/s]
100%|██████████| 4/4 [01:08<00:00, 17.19s/it]
univariate/constant: 100000it [00:16, 6126.66it/s]
univariate/linear: 100000it [00:16, 6034.41it/s]
univariate/exponential: 100000it [00:16, 5942.23it/s]
univariate/staircase: 100000it [00:16, 6067.03it/s]
100%|██████████| 4/4 [01:06<00:00, 16.62s/it]
multivariate/constant: 100000it [00:19, 5219.79it/s]
multivariate/linear: 100000it [00:19, 5178.44it/s]
multivariate/exponential: 100000it [00:19, 5012.65it/s]
multivariate/staircase: 100000it [00:18, 5269.97it/s]
100%|██████████| 4/4 [01:17<00:00, 19.42s/it]


Prepared training inputs for 1080000 training time series and 120000 validation time series.

Sample time series:
  Target shape: (256,)


In [36]:
output_dir = f"results/finetuned_models/chronos2/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

print(f"Fine-tuning Chronos-2 model. Output directory: {output_dir}")

# Fine-tune the model
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    prediction_length=48,
    num_steps=100_000,
    learning_rate=1e-4,
    batch_size=256,
    logging_steps=100,
    # finetune_mode="lora",
    output_dir=output_dir,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    validation_inputs=validation_inputs,
    eval_steps=100
) 

/tmp/ipykernel_1015917/786704418.py:7: FutureWarning: Fine-tuning support is experimental and may be changed in future versions.
  finetuned_pipeline = pipeline.fit(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
100,0.899700,0.315235
200,0.535300,0.337940
300,0.283800,0.154466
400,0.326700,0.337700
500,0.310000,0.210172
600,0.331100,0.182676
700,0.263300,0.246739
800,0.280700,0.433207
900,0.273700,0.678364
1000,0.339500,0.171417


KeyboardInterrupt: 

In [37]:
output_dir

'results/finetuned_models/chronos2/20260212_112823'

In [ ]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

Results written in 'results/finetuned_models/chronos2/20260212_112823'

In [ ]:
from ts_toolkit.pipeline.visualization import finetuning_curves

finetuning_curves(output_dir)

In [ ]:

from ts_toolkit.pipeline.runner import TsPipeline
from ts_toolkit.pipeline.utils import load_config
from glob import glob
import os

ft_experiment = "Bias"
output_paths = {}

for multivariate in [True]:
    multivariate_str = "Multivariate" if multivariate else "Univariate"
    model_repo = f"results/finetuned_models/chronos2/20260212_112823"
    config_dict = {
        "experiment_name": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime",

        "paths": {
            "storage_path": "data/simpletime/datasets_100",
            "output_path": "results"
        },
        "forecaster": {
            "name": "MedianEnsemble",
            "max_length": 512,
            "models": [
                {
                    "name": "Chronos2",
                    "params": {
                        "repo_id": f"{model_repo}/finetuned-ckpt",
                        "batch_size": 64
                    },
                }
            ],
            "alias": f"Chronos2-FineTuned-{ft_experiment}-{multivariate_str}-SimpleTime"
        },
        "evaluation": {
            "to_univariate": not multivariate,
            "show_plot": False,
            "eval_substr": "dim0"
        }
    }
    
    config = load_config(config_dict=config_dict)
    # --- Set up the evaluation pipeline ---
    pipeline = TsPipeline(config)
    pipeline.run(skip_existing=True, num_plots=1)

    pipeline.get_results()
    output_paths[f"Chronos2 (Finetuned - {ft_experiment}) - {multivariate_str}"] = pipeline.output_path
    


Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 782.01it/s]
100it [00:00, 822.46it/s]
100it [00:00, 799.79it/s]
100it [00:00, 774.72it/s]
100it [00:00, 775.63it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
100it [00:00, 764.23it/s]
100it [00:00, 776.82it/s]
100it [00:00, 800.49it/s]
100it [00:00, 797.31it/s]
100it [00:00, 821.03it/s]
100it [00:00, 792.36it/s]
100it [00:00, 802.59it/s]
100it [00:00, 839.21it/s]
100it [00:00, 844.13it/s]
100it [00:00, 816.23it/s]
100it [00:00, 855.12it/s]
100it [00:00, 778.50it/s]
100it [00:00, 761.65it/s]
100it [00:00, 732.00it/s]
100it [00:00, 779.58it/s]
100it [00:00, 737.54it/s]
100it [00:00, 837.40it/s]
100it [00:00, 817.49it/s]
100it [00:00, 822.96it/s]
100it [00:00, 827.15it/s]
100it [00:00, 787.91it/s]
100it [00:00, 797.21it/s]
100it [00:00, 764.19it/s]
100it [00:00, 788.79it/s]
100it [00:00, 847.64it/s]
100it [00:00, 852.42it

Evaluating Datasets:   0%|          | 0/108 [00:00<?, ?it/s]

100it [00:00, 885.01it/s]
100it [00:00, 796.79it/s]
100it [00:00, 830.50it/s]
100it [00:00, 830.10it/s]
100it [00:00, 754.16it/s]
/home/nafiseh/GitHub/timeseries-research/.venv/lib/python3.11/site-packages/gluonts/ev/aggregations.py:123: RuntimeWarning: invalid value encountered in divide
  return self.partial_result / self.n
100it [00:00, 845.23it/s]
100it [00:00, 838.13it/s]
100it [00:00, 799.77it/s]
100it [00:00, 824.66it/s]
100it [00:00, 873.09it/s]
100it [00:00, 880.14it/s]
100it [00:00, 800.23it/s]
100it [00:00, 863.64it/s]
100it [00:00, 812.79it/s]
100it [00:00, 859.59it/s]
100it [00:00, 843.78it/s]
100it [00:00, 726.69it/s]
100it [00:00, 849.23it/s]
100it [00:00, 849.43it/s]
100it [00:00, 882.31it/s]
100it [00:00, 820.44it/s]
100it [00:00, 873.22it/s]
100it [00:00, 810.70it/s]
100it [00:00, 828.37it/s]
100it [00:00, 801.94it/s]
100it [00:00, 764.65it/s]
100it [00:00, 833.05it/s]
100it [00:00, 806.56it/s]
100it [00:00, 776.91it/s]
100it [00:00, 855.65it/s]
100it [00:00, 866.31it

In [39]:
output_paths

{'Chronos2 (Finetuned - Bias) - Multivariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-Bias-Multivariate-SimpleTime_20260212-154222',
 'Chronos2 (Finetuned - Bias) - Univariate': '/home/nafiseh/GitHub/timeseries-research/results/Chronos2-FineTuned-Bias-Univariate-SimpleTime_20260212-154802'}